# Chapter 1 &mdash; Context-Sensitive Patterns: a Copy, Not a Reversal

**Concept 13 of the Chapter 1 decomposition:** *Pattern Class III — Context-Sensitive Patterns*

A function prototype must match its definition. Abstractly that is $ww$ &mdash; and a stack cannot do it, because a stack hands things back <b>reversed</b>.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1-Intro/Concept-Context-Sensitive-Patterns/Concept-Context-Sensitive-Patterns.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_TM         import *
from jove.AnimateTM      import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateTM as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateTM, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


`char func(int, float);` must agree with `char func(int a, float b) { body }` &mdash; the same
list of types, **in the same order**.

Abstracted to bits, that is $ww$: a pattern followed by a **copy**, with no reversal.

* $ww^R$ &mdash; palindrome &mdash; **context-free**. A stack gives you reversal for free.
* $ww$ &nbsp;&nbsp; &mdash; copy &nbsp;&nbsp;&mdash; **not** context-free. A stack cannot replay in order.

This is the sharpest single illustration of why the hierarchy is strict.

## 2. Definitions

### Why a stack fails on $ww$

Push all of $w$, then pop: you get $w$ **backwards**. That is exactly wrong for a copy
and exactly right for a palindrome.

In [ ]:
def stack_replay(w):
    stack = []
    for ch in w:
        stack.append(ch)          # push
    out = ''
    while stack:
        out += stack.pop()        # pop -- comes back reversed
    return out

w = '0110100'
print("w              =", w)
print("stack replay   =", stack_replay(w), " <- reversed, so it matches w w^R")
print("but we need    =", w,               " <- a COPY, for w w")

### The prototype/definition problem, in miniature

In [ ]:
def types_of(decl):
    inside = decl[decl.index('(')+1:decl.index(')')]
    return [t.strip().split()[0] for t in inside.split(',') if t.strip()]

def prototype_matches(proto, defn):
    return types_of(proto) == types_of(defn)

### A Turing machine for $w\#w$

With a separator the job becomes deterministic: mark a symbol on the left, walk right
past `#`, check and mark the matching one, walk back. A TM can do this because it can
**revisit** the tape &mdash; which a stack cannot.

In [ ]:
wpw = md2mc('''TM
I     : 0 ; X , R -> Go0
I     : 1 ; Y , R -> Go1
I     : # ; # , R -> ChkE
Go0   : 0 ; 0 , R -> Go0
Go0   : 1 ; 1 , R -> Go0
Go0   : # ; # , R -> Sk0
Sk0   : X ; X , R -> Sk0
Sk0   : Y ; Y , R -> Sk0
Sk0   : 0 ; X , L -> Back
Go1   : 0 ; 0 , R -> Go1
Go1   : 1 ; 1 , R -> Go1
Go1   : # ; # , R -> Sk1
Sk1   : X ; X , R -> Sk1
Sk1   : Y ; Y , R -> Sk1
Sk1   : 1 ; Y , L -> Back
Back  : 0 ; 0 , L -> Back
Back  : 1 ; 1 , L -> Back
Back  : X ; X , L -> Back
Back  : Y ; Y , L -> Back
Back  : # ; # , L -> Back2
Back2 : 0 ; 0 , L -> Back2
Back2 : 1 ; 1 , L -> Back2
Back2 : X ; X , R -> I
Back2 : Y ; Y , R -> I
Back2 : . ; . , R -> I
ChkE  : X ; X , R -> ChkE
ChkE  : Y ; Y , R -> ChkE
ChkE  : . ; . , S -> Fin
''')
print("w#w TM states :", len(wpw["Q"]))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch1&nbsp;12.&nbsp;Pattern Class II — Context-Free Patterns](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1-Intro/Concept-Context-Free-Patterns/Concept-Context-Free-Patterns.ipynb) &nbsp;&middot;&nbsp; [**Chapter 1** index](https://github.com/ganeshutah/Jove/blob/master/Chapter1-Intro/README.md) &nbsp;&middot;&nbsp; [Ch1&nbsp;14.&nbsp;Pattern Class IV — Recursively Enumerable (Turing-Recognizable) Patterns](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1-Intro/Concept-Recursively-Enumerable-Patterns/Concept-Recursively-Enumerable-Patterns.ipynb)&nbsp;&rarr;

---

## 3. Tests

Prototype matching, on the book's example.

In [ ]:
proto = "char func(int, float);"
good  = "char func(int a, float b) { body }"
bad   = "char func(float a, int b) { body }"
print("types in prototype :", types_of(proto))
assert types_of(proto) == ["int", "float"]
print("matches good defn  :", prototype_matches(proto, good))
print("matches bad  defn  :", prototype_matches(proto, bad), " <- order matters")
assert prototype_matches(proto, good) and not prototype_matches(proto, bad)

Run the TM on $w\#w$. It accepts a genuine copy and rejects a mismatch.

In [ ]:
explore_tm(wpw, "001#001", 200)

Where each pattern class lives.

In [ ]:
print("REGULAR          : a b a b a b ...      no memory of how many")
print("CONTEXT-FREE     : w w^R  (palindrome)  a stack, read back reversed")
print("CONTEXT-SENSITIVE: w w    (a copy)      needs to re-read -- LBA / TM")

## 4. Animation


Step the $w\#w$ machine. Watch it mark a symbol (`0` becomes `X`), cross the `#`,
find the partner, and walk back. **Re-reading the tape is the power a stack lacks.**

In [ ]:
from jove.AnimateTM import *
AnimateTM(wpw, FuseEdges=True)

## 5. Exercises


1. Run `explore_tm(wpw, "01#10", 200)`. It should reject. Trace *where* it gets stuck.
2. Why is $ww$ harder than $w\#w$? (Hint: without `#` you must *guess* the midpoint &mdash;
   Chapter 13 builds a nondeterministic TM for exactly this.)
3. Give a real programming-language rule, other than prototypes, that is
   context-sensitive.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter1-Intro/Concept-Context-Sensitive-Patterns')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')